# Лабораторная работа №1: Визуализация данных
## Набор данных: Iris (鸢尾花)

### 📌 Раздел 1. Текстовое описание набора данных

**Набор данных Iris** (также известный как *Fisher's Iris dataset*) — классический многомерный набор данных, впервые использованный Рональдом Фишером в 1936 году. Датасет стал «Hello World» в машинном обучении.

**Структура:**
- 150 образцов цветков ириса трёх видов (по 50 каждого): *setosa*, *versicolor*, *virginica*.
- 4 признака (в см): длина чашелистика, ширина чашелистика, длина лепестка, ширина лепестка.

**Преимущества для лабораторной:**
- Отсутствие пропусков.
- Компактный размер (150 строк).
- Числовые признаки + категориальная целевая переменная.
- Хорошо изучен, легко интерпретировать результаты.

### 📌 Раздел 2. Основные характеристики датасета

In [ ]:
# ЯЧЕЙКА 1: ИМПОРТ БИБЛИОТЕК
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.datasets import load_iris
from sklearn.preprocessing import LabelEncoder
from scipy import stats

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.float_format', '{:.4f}'.format)

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['legend.fontsize'] = 11

COLORS = {
    'setosa': '#FF6B6B',
    'versicolor': '#4ECDC4',
    'virginica': '#45B7D1'
}

print("✅ Все библиотеки успешно загружены!")

In [ ]:
# ЯЧЕЙКА 2: ЗАГРУЗКА ДАННЫХ
iris = load_iris()
df = pd.DataFrame(data=iris.data, columns=iris.feature_names)
df['species'] = iris.target
df['species_name'] = df['species'].map(dict(enumerate(iris.target_names)))

feature_names_ru = {
    'sepal length (cm)': 'Длина чашелистика, см',
    'sepal width (cm)': 'Ширина чашелистика, см',
    'petal length (cm)': 'Длина лепестка, см',
    'petal width (cm)': 'Ширина лепестка, см'
}
df_renamed = df.rename(columns=feature_names_ru)

print("Первые 5 строк:")
display(df.head())

In [ ]:
# ЯЧЕЙКА 3: ОБЩАЯ ИНФОРМАЦИЯ
print(f"Размер: {df.shape[0]} строк, {df.shape[1]} столбцов")
print(f"Признаков: {len(iris.feature_names)}, классов: {len(iris.target_names)}")
print("Типы данных и пропуски:")
info_df = pd.DataFrame({
    'Тип данных': df.dtypes,
    'Уникальных': df.nunique(),
    'Пропуски': df.isnull().sum(),
    'Пропуски, %': (df.isnull().sum() / len(df) * 100).round(2)
})
display(info_df)
print(f"Дубликатов: {df.duplicated().sum()}")

In [ ]:
# ЯЧЕЙКА 4: СТАТИСТИЧЕСКОЕ ОПИСАНИЕ
display(df.describe())

numeric_cols = iris.feature_names
stats_df = pd.DataFrame({
    'Среднее': df[numeric_cols].mean(),
    'Медиана': df[numeric_cols].median(),
    'Ст.откл.': df[numeric_cols].std(),
    'Асимметрия': df[numeric_cols].skew(),
    'Эксцесс': df[numeric_cols].kurtosis()
})
display(stats_df.round(4))

In [ ]:
# ЯЧЕЙКА 5: РАСПРЕДЕЛЕНИЕ ПО КЛАССАМ
class_counts = df['species_name'].value_counts()
print(class_counts)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors_bar = [COLORS['setosa'], COLORS['versicolor'], COLORS['virginica']]
bars = axes[0].bar(class_counts.index, class_counts.values, color=colors_bar, edgecolor='black', linewidth=1.5)
axes[0].set_title('Распределение по видам')
axes[0].set_ylabel('Количество')
for bar, count in zip(bars, class_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, str(count), ha='center', va='bottom')

axes[1].pie(class_counts.values, labels=class_counts.index, autopct='%1.1f%%', colors=colors_bar, startangle=90, explode=(0.05,0.05,0.05), shadow=True)
axes[1].set_title('Доля классов')
plt.tight_layout()
plt.show()

### 📌 Раздел 3. Визуальное исследование датасета

#### 3.1 Одномерный анализ

In [ ]:
# ЯЧЕЙКА 6: ГИСТОГРАММЫ
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()
colors_hist = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4']
for i, (feature, color) in enumerate(zip(iris.feature_names, colors_hist)):
    sns.histplot(data=df, x=feature, kde=True, color=color, edgecolor='black', alpha=0.7, ax=axes[i], bins=15)
    mean_val = df[feature].mean()
    median_val = df[feature].median()
    axes[i].axvline(mean_val, color='darkred', linestyle='--', linewidth=2, label=f'Среднее = {mean_val:.2f}')
    axes[i].axvline(median_val, color='darkblue', linestyle='-.', linewidth=2, label=f'Медиана = {median_val:.2f}')
    axes[i].set_title(f'Распределение: {feature}')
    axes[i].legend()
plt.suptitle('Гистограммы распределения признаков', fontsize=18)
plt.tight_layout()
plt.show()

In [ ]:
# ЯЧЕЙКА 7: БОКСПЛОТЫ
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()
for i, feature in enumerate(iris.feature_names):
    sns.boxplot(data=df, x='species_name', y=feature, palette=[COLORS['setosa'], COLORS['versicolor'], COLORS['virginica']], ax=axes[i])
    sns.swarmplot(data=df, x='species_name', y=feature, color='black', alpha=0.4, size=3, ax=axes[i])
    axes[i].set_title(f'Боксплот: {feature}')
plt.suptitle('Боксплоты признаков по видам', fontsize=18)
plt.tight_layout()
plt.show()

In [ ]:
# ЯЧЕЙКА 8: СКРИПИЧНЫЕ ДИАГРАММЫ
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()
for i, feature in enumerate(iris.feature_names):
    sns.violinplot(data=df, x='species_name', y=feature, palette=[COLORS['setosa'], COLORS['versicolor'], COLORS['virginica']], ax=axes[i], inner='quartile')
    axes[i].set_title(f'Скрипичная: {feature}')
plt.suptitle('Скрипичные диаграммы', fontsize=18)
plt.tight_layout()
plt.show()

In [ ]:
# ЯЧЕЙКА 9: ГРАФИКИ ПЛОТНОСТИ (KDE)
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()
for i, feature in enumerate(iris.feature_names):
    for species, color in zip(iris.target_names, [COLORS['setosa'], COLORS['versicolor'], COLORS['virginica']]):
        subset = df[df['species_name'] == species]
        sns.kdeplot(data=subset, x=feature, color=color, label=species, ax=axes[i], linewidth=2.5, fill=True, alpha=0.3)
    axes[i].set_title(f'Плотность: {feature}')
    axes[i].legend()
plt.suptitle('Графики плотности распределения', fontsize=18)
plt.tight_layout()
plt.show()

In [ ]:
# ЯЧЕЙКА 10: ECDF
def ecdf(data):
    x = np.sort(data)
    y = np.arange(1, len(x)+1) / len(x)
    return x, y

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()
for i, feature in enumerate(iris.feature_names):
    for species, color in zip(iris.target_names, [COLORS['setosa'], COLORS['versicolor'], COLORS['virginica']]):
        subset = df[df['species_name'] == species][feature]
        x, y = ecdf(subset)
        axes[i].plot(x, y, marker='.', linestyle='none', color=color, label=species, alpha=0.7)
    axes[i].set_title(f'ECDF: {feature}')
    axes[i].legend()
plt.suptitle('Эмпирические функции распределения', fontsize=18)
plt.tight_layout()
plt.show()

#### 3.2 Двумерный анализ

In [ ]:
# ЯЧЕЙКА 11: PAIRPLOT
plot_df = df.copy()
plot_df.columns = ['Длина чашелистика', 'Ширина чашелистика', 'Длина лепестка', 'Ширина лепестка', 'species', 'Вид']
g = sns.pairplot(plot_df, hue='Вид', palette=[COLORS['setosa'], COLORS['versicolor'], COLORS['virginica']],
                 diag_kind='kde', plot_kws={'alpha':0.7, 's':60}, diag_kws={'fill':True})
g.fig.suptitle('Матрица диаграмм рассеяния', fontsize=18, y=1.02)
plt.show()

In [ ]:
# ЯЧЕЙКА 12: JOINTPLOT
g = sns.jointplot(data=df, x='petal length (cm)', y='petal width (cm)', hue='species_name',
                  palette=[COLORS['setosa'], COLORS['versicolor'], COLORS['virginica']],
                  kind='scatter', height=9, ratio=4, space=0.2, marginal_kws={'fill':True, 'alpha':0.5})
g.fig.suptitle('Длина vs Ширина лепестка', fontsize=16, y=1.02)
plt.show()

In [ ]:
# ЯЧЕЙКА 13: РЕГРЕССИОННЫЕ ГРАФИКИ
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.flatten()
pairs = [('sepal length (cm)', 'sepal width (cm)'),
         ('petal length (cm)', 'petal width (cm)'),
         ('sepal length (cm)', 'petal length (cm)'),
         ('sepal width (cm)', 'petal width (cm)')]
for i, (x_feat, y_feat) in enumerate(pairs):
    sns.regplot(data=df, x=x_feat, y=y_feat, ax=axes[i], scatter_kws={'alpha':0.5, 's':40}, line_kws={'color':'red'}, ci=95)
    corr = df[x_feat].corr(df[y_feat])
    axes[i].text(0.05, 0.95, f'r = {corr:.3f}', transform=axes[i].transAxes, fontsize=12,
                 bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    axes[i].set_title(f'{x_feat} vs {y_feat}')
plt.suptitle('Регрессионные графики', fontsize=18)
plt.tight_layout()
plt.show()

In [ ]:
# ЯЧЕЙКА 14: HEXBIN И 2D ГИСТОГРАММА
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
hb = axes[0].hexbin(df['petal length (cm)'], df['petal width (cm)'], gridsize=20, cmap='YlOrRd', edgecolor='white')
axes[0].set_title('Hexbin plot')
plt.colorbar(hb, ax=axes[0], label='Количество')
h2d = axes[1].hist2d(df['petal length (cm)'], df['petal width (cm)'], bins=20, cmap='Blues', cmin=1)
axes[1].set_title('2D гистограмма')
plt.colorbar(h2d[3], ax=axes[1], label='Количество')
plt.suptitle('Плотность точек в 2D', fontsize=16)
plt.tight_layout()
plt.show()

#### 3.3 Многомерный анализ

In [ ]:
# ЯЧЕЙКА 15: RADVIZ
from pandas.plotting import radviz
fig, ax = plt.subplots(figsize=(12,10))
radviz_data = df.copy()
radviz_data['species_name'] = df['species_name']
radviz(radviz_data, 'species_name', color=[COLORS['setosa'], COLORS['versicolor'], COLORS['virginica']], ax=ax, alpha=0.7, s=80)
ax.set_title('RadViz')
plt.show()

In [ ]:
# ЯЧЕЙКА 16: ANDREWS CURVES
from pandas.plotting import andrews_curves
fig, ax = plt.subplots(figsize=(14,8))
andrews_curves(df, 'species_name', color=[COLORS['setosa'], COLORS['versicolor'], COLORS['virginica']], ax=ax, alpha=0.6)
ax.set_title('Andrews Curves')
plt.show()

In [ ]:
# ЯЧЕЙКА 17: PARALLEL COORDINATES
from pandas.plotting import parallel_coordinates
fig, ax = plt.subplots(figsize=(14,8))
norm_df = df.copy()
for col in iris.feature_names:
    norm_df[col] = (df[col] - df[col].min()) / (df[col].max() - df[col].min())
parallel_coordinates(norm_df, 'species_name', color=[COLORS['setosa'], COLORS['versicolor'], COLORS['virginica']], ax=ax, alpha=0.7)
ax.set_title('Parallel Coordinates')
plt.show()

In [ ]:
# ЯЧЕЙКА 18: 3D SCATTER
from mpl_toolkits.mplot3d import Axes3D
fig = plt.figure(figsize=(12,10))
ax = fig.add_subplot(111, projection='3d')
species_colors = df['species_name'].map({'setosa':COLORS['setosa'], 'versicolor':COLORS['versicolor'], 'virginica':COLORS['virginica']})
sc = ax.scatter(df['sepal length (cm)'], df['sepal width (cm)'], df['petal length (cm)'], c=species_colors, s=60, alpha=0.8, edgecolors='white')
ax.set_xlabel('Длина чашелистика')
ax.set_ylabel('Ширина чашелистика')
ax.set_zlabel('Длина лепестка')
ax.set_title('3D Scatter Plot')
legend_elements = [mpatches.Patch(color=COLORS['setosa'], label='setosa'),
                   mpatches.Patch(color=COLORS['versicolor'], label='versicolor'),
                   mpatches.Patch(color=COLORS['virginica'], label='virginica')]
ax.legend(handles=legend_elements)
plt.show()

In [ ]:
# ЯЧЕЙКА 19: FACETGRID
g = sns.FacetGrid(df, col='species_name', hue='species_name', palette=[COLORS['setosa'], COLORS['versicolor'], COLORS['virginica']], height=5, aspect=1.2, col_wrap=3)
g.map(sns.scatterplot, 'sepal length (cm)', 'sepal width (cm)', s=60, alpha=0.7)
g.map(sns.regplot, 'sepal length (cm)', 'sepal width (cm)', scatter=False, color='black', ci=None)
g.add_legend()
g.fig.suptitle('FacetGrid: зависимости по видам', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ЯЧЕЙКА 20: CLUSTER MAP
cluster_data = df[numeric_cols].copy()
cluster_data.index = df['species_name'] + '_' + df.index.astype(str)
g = sns.clustermap(cluster_data, method='ward', metric='euclidean', cmap='RdBu_r', standard_scale=1,
                   figsize=(14,12), dendrogram_ratio=0.15, cbar_pos=(0.02,0.8,0.03,0.18), linewidths=0.5)
g.fig.suptitle('Cluster Map: иерархическая кластеризация', fontsize=16, y=1.02)
plt.show()

### 📌 Раздел 4. Информация о корреляции признаков

In [ ]:
# ЯЧЕЙКА 21: МАТРИЦА КОРРЕЛЯЦИИ
corr_matrix = df[numeric_cols].corr()
print("Матрица корреляции Пирсона:")
display(corr_matrix.round(4))

corr_spearman = df[numeric_cols].corr(method='spearman')
print("\nМатрица корреляции Спирмена:")
display(corr_spearman.round(4))

In [ ]:
# ЯЧЕЙКА 22: ТЕПЛОВАЯ КАРТА
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
corr_display = corr_matrix.copy()
corr_display.columns = ['Дл. чаш.', 'Шир. чаш.', 'Дл. леп.', 'Шир. леп.']
corr_display.index = ['Дл. чаш.', 'Шир. чаш.', 'Дл. леп.', 'Шир. леп.']
fig, axes = plt.subplots(1, 2, figsize=(16,7))
sns.heatmap(corr_display, annot=True, fmt='.3f', cmap='RdBu_r', square=True, linewidths=0.5, mask=mask, vmin=-1, vmax=1, ax=axes[0])
axes[0].set_title('Тепловая карта (верхний треугольник скрыт)')
sns.heatmap(corr_display, annot=True, fmt='.3f', cmap='RdBu_r', square=True, linewidths=0.5, vmin=-1, vmax=1, ax=axes[1])
axes[1].set_title('Полная тепловая карта')
plt.suptitle('Тепловые карты корреляции', fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
# ЯЧЕЙКА 23: КОРРЕЛЯЦИЯ С ЦЕЛЕВОЙ ПЕРЕМЕННОЙ
le = LabelEncoder()
df['species_encoded'] = le.fit_transform(df['species_name'])
corr_with_target = df[numeric_cols + ['species_encoded']].corr()['species_encoded'].drop('species_encoded')
corr_with_target = corr_with_target.sort_values(ascending=False)
print("Корреляция признаков с видом ириса:")
print(corr_with_target)

fig, axes = plt.subplots(1, 2, figsize=(14,6))
colors_corr = ['#4ECDC4' if x>0 else '#FF6B6B' for x in corr_with_target.values]
bars = axes[0].barh(corr_with_target.index, corr_with_target.values, color=colors_corr, edgecolor='black')
axes[0].axvline(0, color='black')
axes[0].set_title('Корреляция с целевой переменной (barh)')
for bar, val in zip(bars, corr_with_target.values):
    axes[0].text(val + 0.03 if val>=0 else val-0.08, bar.get_y()+bar.get_height()/2, f'{val:.3f}', va='center')

axes[1].scatter(corr_with_target.values, corr_with_target.index, s=200, c=colors_corr, edgecolors='black')
axes[1].axvline(0, color='black')
for i, (val, name) in enumerate(zip(corr_with_target.values, corr_with_target.index)):
    axes[1].text(val + 0.03 if val>=0 else val-0.08, i, f'{val:.3f}', va='center')
axes[1].set_title('Корреляция с целевой переменной (scatter)')
plt.suptitle('Корреляция признаков с видом ириса', fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
# ЯЧЕЙКА 24: МАТРИЦА КОРРЕЛЯЦИИ С P-VALUE
def corr_pvalue(df, columns):
    n = len(columns)
    corr = np.zeros((n,n))
    pval = np.zeros((n,n))
    for i in range(n):
        for j in range(n):
            corr[i,j], pval[i,j] = stats.pearsonr(df[columns[i]], df[columns[j]])
    return pd.DataFrame(corr, index=columns, columns=columns), pd.DataFrame(pval, index=columns, columns=columns)

corr_mat, pval_mat = corr_pvalue(df, numeric_cols)
print("p-value для корреляций:")
display(pval_mat.round(4))

fig, ax = plt.subplots(figsize=(10,8))
sig_mask = pval_mat >= 0.05
sns.heatmap(corr_mat, annot=True, fmt='.3f', cmap='RdBu_r', square=True, linewidths=0.5, mask=sig_mask, ax=ax)
for i in range(len(corr_mat)):
    for j in range(len(corr_mat)):
        if pval_mat.iloc[i,j] < 0.001:
            ax.text(j+0.5, i+0.8, '***', ha='center', va='center', color='white')
        elif pval_mat.iloc[i,j] < 0.01:
            ax.text(j+0.5, i+0.8, '**', ha='center', va='center', color='white')
        elif pval_mat.iloc[i,j] < 0.05:
            ax.text(j+0.5, i+0.8, '*', ha='center', va='center', color='white')
ax.set_title('Корреляционная матрица с указанием значимости (* p<0.05, ** p<0.01, *** p<0.001)')
plt.show()

In [ ]:
# ЯЧЕЙКА 25: АНАЛИЗ ВЫБРОСОВ
def mahalanobis_distance(x, data):
    x_minus_mu = x - np.mean(data, axis=0)
    cov = np.cov(data, rowvar=False)
    inv_cov = np.linalg.pinv(cov)
    left = np.dot(x_minus_mu, inv_cov)
    return np.dot(left, x_minus_mu.T).diagonal()

data_matrix = df[numeric_cols].values
mahal_dist = mahalanobis_distance(data_matrix, data_matrix)
df['mahalanobis'] = mahal_dist
threshold = stats.chi2.ppf(0.99, df=4)
df['is_outlier'] = df['mahalanobis'] > threshold
print(f"Количество выбросов: {df['is_outlier'].sum()}, порог: {threshold:.3f}")

fig, axes = plt.subplots(1,2, figsize=(14,6))
axes[0].hist(mahal_dist, bins=30, color='skyblue', edgecolor='black')
axes[0].axvline(threshold, color='red', linestyle='--', label=f'Порог={threshold:.2f}')
axes[0].set_title('Распределение расстояний Махаланобиса')
axes[0].legend()
colors_out = ['red' if o else 'blue' for o in df['is_outlier']]
axes[1].scatter(df['petal length (cm)'], df['petal width (cm)'], c=colors_out, s=60, alpha=0.7, edgecolors='black')
axes[1].set_title('Выбросы на графике Длина vs Ширина лепестка')
plt.suptitle('Анализ выбросов (расстояние Махаланобиса)', fontsize=16)
plt.show()

### 📌 Раздел 5. Дополнительные визуализации (интерактивные)

In [ ]:
# ЯЧЕЙКА 26: ИНТЕРАКТИВНАЯ 3D ВИЗУАЛИЗАЦИЯ (PLOTLY)
import plotly.express as px
fig = px.scatter_3d(df, x='sepal length (cm)', y='sepal width (cm)', z='petal length (cm)',
                    color='species_name', color_discrete_map={'setosa':COLORS['setosa'], 'versicolor':COLORS['versicolor'], 'virginica':COLORS['virginica']},
                    size='petal width (cm)', size_max=15, opacity=0.8,
                    title='Интерактивная 3D диаграмма рассеяния')
fig.update_layout(scene=dict(xaxis_title='Длина чашелистика', yaxis_title='Ширина чашелистика', zaxis_title='Длина лепестка'))
fig.show()

In [ ]:
# ЯЧЕЙКА 27: ИНТЕРАКТИВНАЯ МАТРИЦА РАССЕЯНИЯ (PLOTLY)
fig = px.scatter_matrix(df, dimensions=numeric_cols, color='species_name',
                        color_discrete_map={'setosa':COLORS['setosa'], 'versicolor':COLORS['versicolor'], 'virginica':COLORS['virginica']},
                        title='Интерактивная матрица диаграмм рассеяния', height=800, width=800)
fig.update_traces(diagonal_visible=False)
fig.show()

In [ ]:
# ЯЧЕЙКА 28: СВОДНАЯ ТАБЛИЦА ВИЗУАЛИЗАЦИЙ
visualizations = [
    ("Гистограммы", "Распределение признаков", "matplotlib/seaborn"),
    ("Боксплоты", "Распределение по классам", "seaborn"),
    ("Скрипичные диаграммы", "Плотность по классам", "seaborn"),
    ("Графики плотности (KDE)", "Сглаженные распределения", "seaborn"),
    ("ECDF", "Эмпирические функции", "matplotlib"),
    ("Матрица рассеяния", "Попарные зависимости", "seaborn"),
    ("Jointplot", "Двумерное распределение", "seaborn"),
    ("Регрессионные графики", "Линейные зависимости", "seaborn"),
    ("Hexbin / 2D гистограмма", "Плотность точек", "matplotlib"),
    ("RadViz", "Радиальная визуализация", "pandas"),
    ("Andrews Curves", "Кривые Эндрюса", "pandas"),
    ("Parallel Coordinates", "Параллельные координаты", "pandas"),
    ("3D Scatter", "Трёхмерная диаграмма", "matplotlib"),
    ("FacetGrid", "Многопанельные графики", "seaborn"),
    ("Cluster Map", "Иерархическая кластеризация", "seaborn"),
    ("Тепловая карта корреляции", "Матрица корреляции", "seaborn"),
    ("Корреляция с целевой", "Важность признаков", "matplotlib"),
    ("Матрица с p-value", "Статистическая значимость", "matplotlib"),
    ("Анализ выбросов", "Расстояние Махаланобиса", "matplotlib"),
    ("Интерактивная 3D", "Plotly Express", "plotly")
]
viz_df = pd.DataFrame(visualizations, columns=["Тип", "Описание", "Библиотека"])
display(viz_df)
print(f"Всего визуализаций: {len(visualizations)}")

### 📌 Раздел 6. Выводы и заключение

In [ ]:
# ЯЧЕЙКА 29: ОСНОВНЫЕ ВЫВОДЫ
print("""
1. СТРУКТУРА ДАННЫХ: 150 образцов, 4 признака, 3 класса, пропуски отсутствуют.
2. РАСПРЕДЕЛЕНИЕ: длина и ширина лепестка имеют бимодальное распределение; Setosa заметно отличается.
3. ВЗАИМОСВЯЗИ: сильная положительная корреляция между длиной и шириной лепестка (≈0.96), между длиной лепестка и длиной чашелистика (≈0.87).
4. РАЗДЕЛИМОСТЬ: Setosa легко отделяется; Versicolor и Virginica частично пересекаются.
5. КОРРЕЛЯЦИЯ С ЦЕЛЕВОЙ: наибольшая — ширина лепестка (≈0.96), длина лепестка (≈0.95); наименьшая — ширина чашелистика (≈-0.42).
6. ВЫБРОСЫ: обнаружено несколько потенциальных выбросов, они не критичны.
""")

In [ ]:
# ЯЧЕЙКА 30: ЭКСПОРТ РЕЗУЛЬТАТОВ
df.to_csv('iris_analysis.csv', index=False)
corr_matrix.to_csv('correlation_matrix.csv')
print("✅ Данные сохранены в CSV.")
print(f"Дата выполнения: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("✅ Лабораторная работа успешно выполнена!")